# Round 3, RUN B — the same recipe, EVERY hand-verified real strip

⚠ **This is still ROUND 3** (owner, 2026-09-01). The exam was already read once this round, on
`r3-final-stage2-last`; choosing among these runs is `_realval_v2`'s job, and a second exam read
is a rule question for the owner, not something this notebook assumes.

**One variable against RUN A: the real training data.** Run A trains on `strips_b8` alone. Run B
adds **`strips_oldhuman`** — 1,408 strips that a human verified by hand, cut by the slicer the app
no longer runs. Everything else is identical to Run A: same corpus (`strips_v7_final`), same
stage 1, same 4,000 stage-2 steps, same lr, same mix. So A vs B answers "does the second cut help?"
and nothing else.

⚠ **The two pools overlap in MUSIC and differ in PIXELS.** Much of `strips_oldhuman`'s music is
already inside `strips_b8` at today's geometry. The owner's argument is that b8 stays in the mix, so
this **adds** a second cut rather than replacing the current one. ⚠ **Nobody has ever scored a model
trained on mixed crop roots — treat the whole run as a lead, not a fix.**

## ⛔ Stage 1 is NOT re-run — it reuses Round 3's

Same reason as Run A: stage 1 is identical work, its checkpoint is on Drive, and re-running it costs
~2.5 h **and** adds a further difference (training is not bit-deterministic on a GPU). Total GPU here
is the stage-2 run alone: at Run A's ~1.6 s/step, 5,000 steps is roughly **2.0–2.3 h**.

## ⚠ `:4`, NOT `:5` — the repeat is the one number that had to move

`:5` was measured for b8 ALONE. Counted on the real manifests (2026-09-01):

| pool | strips | train-side |
|---|---|---|
| `strips_b8` | 3,929 | 3,539 |
| `strips_oldhuman` | 1,408 | 1,238 |
| **combined** | **5,337** | **4,777** |

Against 36,032 train-side synthetic strips: **`:4` puts real at 34.7%** of stage-2 batches, close to
Round 3's recipe (b8 alone at `:5` = 32.9%). **`:5` here would read 39.9%** — that changes the mix as
a side effect, and the mix is not what this run is testing. `train.py` prints the
`real pool … : N train xR / M val strips` line for each pool; read both.

## ⭐ Read the `best-real` checkpoint

`train.py` saves three: `best` (blended val loss, **~92% synthetic by strip count**), **`best-real`**
(the REAL val loss alone), `last` (resume point). ⭐ `best-real` is also the **unattended-run safety
net**: it is written on improvement every 250 steps straight to Drive, so a session that dies at
step 3,000 still leaves a legitimately selected model behind — unlike `last`, which would then be a
mid-cosine checkpoint at a learning rate that never annealed. ⛔ The blend has now picked wrong **twice** — Round 3
stamped `best` at step 500, Run A at step **250**, the first evaluation of stage 2. On Run A only
`best-real` (step 2,500) was usable. Expect the same here.

## After the run

Bring all three home and choose on `_realval_v2` with `paired_arm_score.py`.
⛔ Never choose on the val losses printed here — real-val selects, the exam grades, once.


In [ ]:
# ===== THE ONLY KNOBS IN THIS NOTEBOOK — and the mix is NOT one of them =====
ARM = 'r3b'
STRIPS = 'data/synthetic/strips_v7_final'   # the 3-flag render: staccato + concave tuplet + usul barline
ZIP = 'tnc_round3_finalb_colab.zip'   # ⚠ the finalb zip — it is the one carrying strips_oldhuman
DRIVE = '/content/drive/MyDrive/tnc'
# THE REAL POOLS — TWO of them, and that is Run B's whole variable.
# ⛔ Never add the RAW retired pools (strips_nota / strips_r1 / strips_tup): 922 of strips_nota's
# rows are machine-verdicted. strips_oldhuman is the HAND-VERIFIED subset of those three, 1,408 rows.
REAL = 'data/real/rung3/strips_b8'          # 3,929 strips / 3,539 train-side
REAL2 = 'data/real/rung3/strips_oldhuman'   # 1,408 strips / 1,238 train-side
REPEAT = 4          # -> real 34.7% of stage-2 batches at 4,777 combined train-side. ⚠ NOT 5 (39.9%).
print(ARM, STRIPS, ZIP, f'| real: {REAL}:{REPEAT} + {REAL2}:{REPEAT} | mix: train.py defaults (0.65/0.35, scan off)')


In [ ]:
# Which GPU did we get? (T4 16GB / L4 24GB / A100 40GB)
!nvidia-smi

In [ ]:
# Mount Google Drive (approve the popup).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
# Copy the package Drive -> VM disk and unzip (fast local disk for the dataloader).
# ⛔ CHECK THE COPY BEFORE TRUSTING IT. The Drive FUSE mount drops — "Transport endpoint is not
# connected" — most often when Drive has not finished processing a fresh upload. `cp` then leaves a
# TRUNCATED file, `unzip` fails, and the first error you actually see is a missing render_config.json
# three steps later, which looks like a bad zip and is not. Fail here instead, with the real reason.
import os, subprocess
ZIP_BYTES = 871130067   # `stat -f %z data/colab/tnc_round3_finalb_colab.zip` on the Mac

src = f'{DRIVE}/{ZIP}'
assert os.path.exists(src), (
    f'{src} not on Drive. Either the upload is unfinished or the mount is dead — '
    'Runtime > Restart session, re-run the mount cell, then `!ls -l {DRIVE}` to look.')
have = os.path.getsize(src)
assert have == ZIP_BYTES, (
    f'Drive has {have:,} bytes, expected {ZIP_BYTES:,}. The upload is INCOMPLETE or still syncing — '
    'wait for it to settle and re-run this cell. Do not unzip a partial file.')
print(f'Drive copy looks whole: {have:,} bytes')

# rsync, not cp: it is resumable, so a mount hiccup costs a retry rather than the whole transfer.
!rsync --progress {DRIVE}/{ZIP} /content/
got = os.path.getsize(f'/content/{ZIP}')
assert got == ZIP_BYTES, (
    f'copied {got:,} of {ZIP_BYTES:,} bytes — the mount dropped mid-copy. '
    'Restart the session and try again; re-run this cell, rsync will resume.')

# `unzip -t` reads the central directory, which is exactly what a truncated file lacks.
assert subprocess.run(['unzip', '-tq', f'/content/{ZIP}']).returncode == 0, \
    'the copied zip fails its own integrity test — delete /content/{ZIP} and re-copy'
print('zip integrity OK')

!rm -rf /content/tnc && mkdir /content/tnc
!cd /content/tnc && unzip -q /content/{ZIP}

# WHICH CORPUS IS ACTUALLY ON DISK. All three flags are LABEL-FREE — they change pixels only, so the
# manifest cannot tell this corpus from an arm's. render_config.json is the ONLY place they are
# checkable, which is why they are asserted here as well as in make_round3_colab_zip.sh.
import json
cfg = json.load(open(f'/content/tnc/{STRIPS}/render_config.json'))
print(cfg)
assert cfg.get('staccatoNoise') is True,  'MISSING --staccato-noise — this is not the final render'
assert cfg.get('concaveTuplet') is True,  'MISSING --concave-tuplet — this is not the final render'
assert cfg.get('usulBarline') is True,    'MISSING --usul-barline — this is not the final render'
assert cfg['legacyTupletMark'] is False and cfg['thinSharps'] is True and cfg['printNoise'] is False
assert cfg.get('maxMeasures', None) is None

# WHICH REAL POOLS ARE ON DISK. BOTH must be here — a missing strips_oldhuman would train a
# silent copy of Run A. And the RAW retired pools must still be absent: their filenames survive a
# re-slice and their pixels do not, so a name check is not enough on its own, but their ABSENCE is
# decisive. strips_oldhuman is the hand-verified SUBSET of them, rebuilt as its own pool.
import os
assert os.path.isdir(f'/content/tnc/{REAL}'), f'{REAL} missing from the zip'
assert os.path.isdir(f'/content/tnc/{REAL2}'), (
    f'{REAL2} missing — this is the `final` zip, not `finalb`, and the run would silently repeat '
    'Run A. Rebuild with: sh scripts/make_round3_colab_zip.sh finalb')
for dead in ('strips_nota', 'strips_r1', 'strips_tup'):
    assert not os.path.isdir(f'/content/tnc/data/real/rung3/{dead}'), \
        f'RAW RETIRED POOL {dead} IS IN THE ZIP — rebuild it with: sh scripts/make_round3_colab_zip.sh finalb'
!wc -l /content/tnc/{STRIPS}/manifest.jsonl
!wc -l /content/tnc/{REAL}/manifest.jsonl    # expect 3929
!wc -l /content/tnc/{REAL2}/manifest.jsonl   # expect 1408
!python -c "import json;s=json.load(open('/content/tnc/data/split_v4.json'));print('train',len(s['train_pieces']),'val',len(s['val_pieces']))"

# ⛔ THE ZIP MUST CARRY THE PATCHED train.py. `best-real` (the checkpoint selected on the REAL val
# loss) was added 2026-09-01, and WITHOUT IT a 4,000-step run silently discards its best real-page
# checkpoint between evals — the exact failure Round 3 hit at 2,000 steps. A stale zip would train
# happily and give you the wrong model, so this fails loudly instead.
src = open('/content/tnc/src/vision/train.py').read()
assert 'best_real' in src and 'best-real' in src, (
    "STALE train.py IN THE ZIP — it has no `best-real` checkpoint.\n"
    "Fix: rebuild the zip on the Mac (`sh scripts/make_round3_colab_zip.sh final`) and re-upload,\n"
    "or drop the patched src/vision/train.py into MyDrive/tnc/ and copy it over this one.")
print('train.py: best-real checkpoint present ✅')

In [ ]:
# Dependencies (torch + torchvision are preinstalled on Colab).
!pip -q install transformers albumentations opencv-python-headless

In [ ]:
# ===== THE FLAGS ARE THE RENDER — prove they are in the PIXELS before spending a GPU hour =====
# Nothing downstream records them: labels, manifest and split are what they would be with the flags
# off (188 strip labels over 4 scores are byte-identical with --usul-barline on and off). So the
# check is on the IMAGES, and on the mix being the default.
%cd /content/tnc
import sys
sys.path.insert(0, 'src/vision')
from augment import Augmenter
a = Augmenter(seed=7)
assert (a.photo_share, a.scan_share) == (0.35, 0.0), (a.photo_share, a.scan_share)
print(f'mix: screenshot {1-a.photo_share-a.scan_share:.2f} / photo {a.photo_share} / scan {a.scan_share}  (default)')

import json, random
from PIL import Image
rows = [json.loads(l) for l in open(f'{STRIPS}/manifest.jsonl')]
print(f'{len(rows)} synthetic strips')

# LOOK AT THEM. Each flag is coined PER PIECE, so a sample of one piece shows nothing — these draw
# from strips whose label gives the flag something to act on.
def show(pred, what, n=2):
    hits = [r for r in rows if pred(r)]
    print(f'{what}: {len(hits)} candidate strips')
    random.Random(7).shuffle(hits)
    for r in hits[:n]:
        print('  ', r['image'])
        display(Image.open(f"{STRIPS}/{r['image']}"))

show(lambda r: '.' in r['label'], 'staccato — dots ABOVE/BELOW noteheads, not beside')
show(lambda r: '\\tup3' in r['label'], 'tuplet — some pieces draw a CONTINUOUS arc with the 3 inside it')
show(lambda r: '|' in r['label'], 'usul barline — light DASHED rules INSIDE the bar, at the beat groups')

In [ ]:
# SHAKEOUT (~3 min): 150 tiny steps from BASE — a WIRING smoke, not a result.
# Expect: `vocab: +25 tokens -> 100 ids`, TWO real pools listed, `exam-disjointness OK`,
# `augment=on (screenshot 0.65 / photo 0.35)`, and val loss FALLING.
# ⚠ READ BOTH `real pool ... : N train xR / M val strips` LINES — that is the only place the pools
# and their repeat are visible, and they are what the header's 34.7% rests on.
# ⚠ The shakeout runs the pools at x1; the repeat only appears in the stage-2 cell.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL} --real-dir {REAL2} \
    --every-share 0.15 --out-dir /content/r3-shakeout \
    --lr 3e-5 --warmup-steps 30 --max-steps 150 --batch-size 8 \
    --limit-val 40 --eval-every 50 --save-every 50 --log-every 25 --num-workers 2

In [ ]:
# ===== CALIBRATE THROUGHPUT ON *THIS* RUNTIME (~2-3 min) — before any long run =====
#   hours = (steps * batch) / samples_per_sec / 3600
# ⚠ Whatever you set, the STEP COUNTS AND BATCH SIZE must match this run's: stage 1 is REUSED
# (6000 @ 16, already spent), and stage 2 here is 5000 @ 16.
# --num-workers and the GPU model do not change the result; those two do.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!nproc
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL} --real-dir {REAL2} \
    --every-share 0.15 --out-dir /content/calib \
    --lr 3e-5 --warmup-steps 20 --max-steps 60 --batch-size 16 \
    --limit-val 8 --eval-every 60 --save-every 60 --log-every 20 --num-workers 10

In [ ]:
# ===== STAGE 1 — ⛔ DO NOT RUN THIS. REUSE ROUND 3's. =====
# Run B's variable is the real DATA, not stage 1 — and stage 1 sees no real strips at all, so it is
# IDENTICAL work to the Round-3 final run and to Run A. That checkpoint is on Drive at
# `r3-final-stage1/best`. Re-running it costs ~2.5 h of GPU.
# ⭐ AND REUSING IT IS THE MORE CORRECT CHOICE, not merely the cheaper one: training is not
# bit-deterministic on a GPU, so a re-run produces a slightly DIFFERENT stage-1 model, and Run B
# would then differ from Run A in two places instead of one. The shared stage-1 checkpoint is what
# makes "b8 alone vs b8 + strips_oldhuman" the only variable.
STAGE1 = f'{DRIVE}/r3-final-stage1/best'
import os
assert os.path.isdir(STAGE1), (
    f'{STAGE1} missing. If Round 3 s stage 1 is genuinely gone, uncomment the cell below and spend '
    'the 2.5 h — but then say so when reporting, because B vs A is no longer a clean pair.')
print('reusing Round 3 stage 1:', STAGE1)

# --- only if the checkpoint above is really gone -------------------------------------------------
# ⚠ NO --real-dir here: stage 1 is the synthetic stage. Adding the real pools would make this a
# different recipe from the one Run A and Round 3 started from.
# %cd /content/tnc
# !python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
#     --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage1 \
#     --lr 3e-5 --max-steps 6000 --batch-size 16 --num-workers 10 \
#     --eval-every 250 --save-every 250


In [ ]:
# ===== STAGE 2 — real-SPECIALISATION fine-tune from stage 1 =====
# Fresh LOW lr + short warmup from the stage-1 checkpoint.
# ⚠ TWO --real-dir flags — that is Run B's variable. Both at `:4`.
# ⚠ 5,000 STEPS (owner, 2026-09-01). Run A ran 4,000, so this is a SECOND difference from Run A and
# a B-vs-A gap is no longer attributable to the data alone — see the header.
# ⛔ Do NOT read Run A's "minimum at step 2,500" as a step number here: the cosine lr hits zero at
# --max-steps, so this is a different schedule and its minimum sits somewhere else. Watch the `real`
# column and let `best-real` do the choosing.
# ⛔ `:5` is what b8 ALONE was measured at (32.9% real). With strips_oldhuman beside it, `:4` reads
# 34.7% and `:5` would read 39.9% — a mix change this run is not testing.
# ⚠ STARTS FROM ROUND 3's STAGE 1 (see the cell above) — that shared start is what keeps this a
# clean one-variable comparison against Run A.
%cd /content/tnc
!python -u src/vision/train.py --model {STAGE1} \
    --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL}:{REPEAT} --real-dir {REAL2}:{REPEAT} \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 5000 --batch-size 16 --num-workers 10 \
    --eval-every 250 --save-every 250


In [ ]:
# RESUME after a disconnect: re-run the setup cells (mount, copy/unzip, deps, and the STAGE 1 cell
# that only CHECKS the checkpoint), then this. --resume reloads model+optimizer+scheduler from
# <out-dir>/last and ignores --model.
# ⚠ This is shaped for STAGE 2, the only stage this notebook runs. The flags must match the stage-2
# cell exactly — including BOTH --real-dir pools, `:4`, and `--max-steps 5000`. A resume at 4000
# would re-anneal the lr on a different curve, which is a different experiment, not a continuation.
%cd /content/tnc
!python -u src/vision/train.py --strips-dir {STRIPS} --split data/split_v4.json \
    --real-dir {REAL}:{REPEAT} --real-dir {REAL2}:{REPEAT} \
    --every-share 0.15 --out-dir {DRIVE}/r3-{ARM}-stage2 \
    --lr 1e-5 --warmup-steps 100 --max-steps 5000 --batch-size 16 --num-workers 10 \
    --eval-every 250 --save-every 250 --resume


In [ ]:
# ===== SANITY ONLY — did anything break? =====
# NOT the pre-registered number, and NOT the selection. Both are read on the Mac. This cell exists so
# a broken run is caught before it is downloaded, and to see `best` and `last` side by side.
%cd /content/tnc
# ⚠ b8 ONLY, deliberately: Run A's sanity cell built this pool from b8, and adding
# strips_oldhuman's val strips would make the two numbers uncomparable. The pre-registered
# read is `_realval_v2` on the Mac either way.
!python src/vision/make_realval_pool.py --real-dir {REAL} --split data/split_v4.json

for ck in [f'r3-{ARM}-stage2/best', f'r3-{ARM}-stage2/best-real', f'r3-{ARM}-stage2/last']:
    print('=' * 70, '\n==', ck)
    !python src/vision/eval_omr.py --checkpoint {DRIVE}/{ck} \
        --strips-dir data/real/rung3/_realval --split none --show-errors 0

## After the run

1. **Download ALL THREE** from `MyDrive/tnc/r3-r3b-stage2/` — `best`, **`best-real`**, `last`.
   ⭐ On both previous runs `best-real` was the only usable one; expect that again.
2. **Compare Run B against RUN A first** — that pair has one variable, the extra pool:
   ```bash
   .venv-ml/bin/python scripts/rung3/paired_arm_score.py \
       --ctl data/checkpoints/r3a-stage2-best-real --arm data/checkpoints/r3b-stage2-best-real \
       --pool data/real/rung3/_realval_v2
   ```
3. **Then against Round 3's shipped choice**, `r3-final-stage2-last`, which is what any of this has
   to beat to matter.
4. ⛔ **The exam is one-shot per round.** It was read for Round 3 on 2026-09-01, and Runs A and B are
   the SAME round. Do not read it again unless the owner re-opens it deliberately.
